# Heart Disease Prediction — End-to-End ML Pipeline
**Dataset:** UCI Heart Disease Dataset (Cleveland Clinic Foundation)  
**Goal:** Binary classification — predict whether a patient has heart disease (1) or not (0)  
**Models:** Logistic Regression · Naive Bayes · KNN · SVM · Random Forest

---

## Section 1 — Imports

We import all required libraries up front:
- **pandas / numpy** — data loading, manipulation, and numerical operations.
- **pathlib.Path** — filesystem path handling (cross-platform portability).
- **sklearn** modules — train/test splitting, feature scaling, all five classifiers, and every evaluation metric.
- `warnings.filterwarnings('ignore')` — suppresses non-critical convergence warnings to keep output readable.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')

## Section 2 — Load Dataset

We read the CSV with `skiprows=1` because the file was exported from Excel, which inserts a `sep=,` metadata line as the very first row. Skipping it ensures pandas reads the real column headers correctly.

The original UCI `target` column uses values **0–4** (0 = no disease, 1–4 = varying severity). We **binarise** it: any value > 0 becomes **1** (disease present), turning this into a clean binary classification problem.

In [ ]:
print("=" * 60)
print("  UCI Heart Disease Dataset — Cleveland Clinic")
# UPDATED: Reading from the file created in the previous step
print("  Loading from: heart_disease_for_excel.csv")
print("=" * 60)

# Use skiprows=1 to ignore the 'sep=,' line at the top of the file
df = pd.read_csv('heart_disease_for_excel.csv', skiprows=1)

# The original UCI target can have values 0-4.
# 0 = no disease, 1-4 = disease present -> binarise to 0/1
df['target'] = (df['target'] > 0).astype(int)

print(f"\nDataset loaded successfully!")
print(f"  Shape  : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  Target : 0 = No Disease, 1 = Has Disease")
print(f"\n  Patients WITH heart disease    : {(df['target'] == 1).sum()}")
print(f"  Patients WITHOUT heart disease : {(df['target'] == 0).sum()}")
print(f"\nFirst 5 rows:")
print(df.head())

## Section 3 — Data Preprocessing & EDA

This section covers all steps needed to clean and understand the data before training:
1. **Missing Values** — detect and fill with column median.
2. **Outlier Detection** — flag extreme values using the IQR method.
3. **Feature Encoding** — confirm all features are already numeric.
4. **Descriptive Statistics** — summarise distributions across all features.
5. **Correlation Analysis** — measure each feature's linear relationship with the target.

### 3a — Handling Missing Values
We fill `NaN` values with the column **median** — more robust than the mean because it is not distorted by extreme outliers.

In [ ]:
print("\n" + "=" * 60)
print("  SECTION 2 — DATA PREPROCESSING & EDA")
print("=" * 60)

# --- Missing Values ---
print("\n--- Handling Missing Values ---")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  No missing values found.")
if missing.sum() > 0:
    df.fillna(df.median(numeric_only=True), inplace=True)
    print("  Filled missing values with column median.")

# --- Outlier Detection (IQR Method) ---
print("\n--- Outlier Detection (IQR Method) ---")
for col in ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f"  {col:<12}: {n_out} outlier(s) detected")

# --- Feature Encoding ---
print("\n--- Feature Encoding ---")
print("  All features are already numeric.")

# --- EDA: Basic Statistics ---
print("\n--- EDA: Dataset Statistics ---")
print(df.describe().round(2))

# --- EDA: Feature Correlation with Target ---
print("\n--- EDA: Feature Correlation with Target ---")
corr = df.corr(numeric_only=True)['target'].drop('target').sort_values(ascending=False)
print(corr.round(4))

## Section 4 — Train / Test Split (80% | 20%)

We separate features (X) and labels (y), then split 80/20:
- `stratify=y` — preserves the same disease/no-disease ratio in both splits, preventing accidental class imbalance.
- `random_state=42` — makes the split deterministic and reproducible across runs.

In [ ]:
print("\n" + "=" * 60)
print("  SECTION 3 — TRAIN / TEST SPLIT")
print("=" * 60)

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\n  Total samples    : {len(df)}")
print(f"  Training samples : {X_train.shape[0]} (80%)")
print(f"  Testing samples  : {X_test.shape[0]} (20%)")

## Section 5 — Feature Scaling (StandardScaler)

**StandardScaler** transforms each feature to mean = 0 and std = 1. This is critical for:
- **KNN** and **SVM** which rely on Euclidean distance — unscaled features would bias towards high-magnitude columns.
- **Logistic Regression** which uses gradient descent and converges faster with scaled inputs.

> `fit_transform` on training data only — then `transform` the test set. Fitting on test data would be **data leakage**.

In [ ]:
print("\n--- Feature Scaling (StandardScaler) ---")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("  StandardScaler applied.")

## Section 6 — Evaluation Helper Function

A reusable function that prints metrics for any model and **returns** a results dictionary for the final comparison table.

| Metric | What it measures |
|---|---|
| **Accuracy** | % of all predictions that are correct |
| **Precision** | Of patients predicted sick, how many truly are? |
| **Recall** | Of all truly sick patients, how many did we catch? |
| **F1-Score** | Harmonic mean of Precision and Recall |

In medical diagnosis, **Recall** is the most important — a False Negative (missed case) is far more dangerous than a False Positive.

In [ ]:
results = []

def evaluate_model(name, y_test, y_pred):
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")

    return {'Model': name, 'Accuracy': round(acc,4), 'Precision': round(prec,4),
            'Recall': round(rec,4), 'F1-Score': round(f1,4)}

## Section 7 — Baseline Models

Baseline models are simple, well-understood algorithms. Their scores set a performance floor — any advanced model should beat them to justify extra complexity.

### 7a — Logistic Regression
Despite its name, this is a **classification** algorithm. It estimates the probability of class 1 via the sigmoid function applied to a weighted linear combination of features. It is fast, interpretable, and a strong baseline.
- `max_iter=1000` — extra iterations ensure convergence.
- `random_state=42` — reproducibility.

In [ ]:
print("\n" + "=" * 60)
print("  SECTION 6 — BASELINE MODELS")
print("=" * 60)

# Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('Logistic Regression', y_test, lr_model.predict(X_test_scaled)))

### 7b — Naive Bayes (Gaussian)

**Gaussian Naive Bayes** applies Bayes' theorem under two assumptions:
1. Features are **conditionally independent** given the class label.
2. Each feature follows a **Gaussian (normal) distribution**.

It trains almost instantly and is a useful benchmark, even though the independence assumption is often partially violated in real data.

In [ ]:
# Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('Naive Bayes', y_test, nb_model.predict(X_test_scaled)))

## Section 8 — Advanced Models

### 8a — K-Nearest Neighbors (k=7)

KNN is a **lazy learner** — it stores all training points and classifies new samples by the majority class among the **k=7 nearest neighbours** in feature space.
- k=7 (odd) avoids tie votes.
- Feature scaling is **mandatory** — KNN uses Euclidean distance, so large-scale features would dominate.

In [ ]:
print("\n" + "=" * 60)
print("  SECTION 7 — ADVANCED MODELS")
print("=" * 60)

# KNN
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('KNN', y_test, knn_model.predict(X_test_scaled)))

### 8b — Support Vector Machine (Linear Kernel, C=1)

SVM finds the **maximum-margin hyperplane** separating both classes in scaled feature space.
- `kernel='linear'` — assumes the classes are approximately linearly separable.
- `C=1` — regularisation: low C widens the margin (more tolerant of misclassification); high C tightens the fit.

In [ ]:
# SVM
svm_model = SVC(kernel='linear', C=1, random_state=42)
svm_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('SVM', y_test, svm_model.predict(X_test_scaled)))

### 8c — Random Forest (100 Trees, max_depth=5)

Random Forest is an **ensemble** method. It trains many decision trees on random subsets of data and features (**bagging**), then combines predictions by majority vote.
- `n_estimators=100` — 100 trees reduce variance without excessive training time.
- `max_depth=5` — caps depth to prevent overfitting.
- Naturally robust to outliers and does not strictly require scaling.

In [ ]:
# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('Random Forest', y_test, rf_model.predict(X_test_scaled)))

## Section 9 — Final Comparative Report

All five result dictionaries are collected into a DataFrame and sorted by **Accuracy** descending. The top row is the best overall model. For clinical deployment, also prioritise the model with the highest **Recall** to minimise missed diagnoses.

In [ ]:
comparison_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)

print("\n\n" + "=" * 72)
print("                    FINAL MODEL COMPARISON REPORT")
print("=" * 72)
print(comparison_df.to_string(index=False))
print("=" * 72)

best_name = comparison_df.iloc[0]['Model']
print(f"\n  Best Model : {best_name}")

## Section 10 — Conclusion

This notebook delivered a complete, reproducible ML pipeline for binary heart disease prediction on the UCI Cleveland dataset.

**Pipeline summary:**
1. **Load** — `skiprows=1` handles the Excel `sep=,` metadata row; target binarised 0–4 → 0/1.
2. **Preprocess** — missing values filled with median; IQR outlier counts reported.
3. **EDA** — descriptive statistics and Pearson correlation identify strongest predictors.
4. **Split** — stratified 80/20 with `random_state=42`.
5. **Scale** — StandardScaler fitted on training data only (prevents leakage).
6. **Train & Evaluate** — five classifiers compared on Accuracy, Precision, Recall, F1.
7. **Compare** — final table sorted by Accuracy; best model identified.

For clinical use, prioritise the model with the highest **Recall** to minimise missed diagnoses.